# Import BAFU EcoSpold 2 with biosphere 3.10

Select the **bw** kernel and run the cells in order. This notebook creates a Brightway project with biosphere 3.10, imports the exported `.spold` files, and writes the inventory when all exchanges link.

The export already contains the approved mappings and excludes 2,991 unsupported biosphere exchanges. Their audit is in `data/processed/ecospold2-biosphere310/audit/excluded-exchanges.jsonl`. All import and uncertainty-preservation code is included below; no local Python helper files are required. The mappings are already in the XML and are not applied again.

## 1. Paths and names

Run from the repository or its `scripts/` folder. Change `PROJECT` or `DATABASE` to create another import. Storage must be configured before importing Brightway; start with a fresh kernel.

In [ ]:
import os
from pathlib import Path

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data/processed").is_dir() and (path / "scripts").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its scripts folder.")

EXPORT = ROOT / "data/processed/ecospold2-biosphere310"
SOURCE = EXPORT / "datasets"
STORAGE = ROOT / "artifacts/brightway"
PROJECT = "bafu-2026-ecospold2-biosphere-310"
BIOSPHERE = "ecoinvent-3.10-biosphere"
DATABASE = "BAFU:2026-reimported"

if not any(SOURCE.glob("*.spold")):
    raise FileNotFoundError(f"No exported EcoSpold 2 files found in {SOURCE}.")

# Configure Brightway storage before importing bw2data or bw2io.
STORAGE.mkdir(parents=True, exist_ok=True)
os.environ["BRIGHTWAY2_DIR"] = str(STORAGE)

## 2. Create or reuse the project

The first run installs Brightway's biosphere 3.10 project archive, using its local cache when available. Internet access is needed only if the archive is not cached.

In [ ]:
import bw2data as bd
import bw2io as bi

if PROJECT not in bd.projects:
    bi.install_project(
        BIOSPHERE,
        project_name=PROJECT,
        projects_config={BIOSPHERE: "ecoinvent-3.10-biosphere.tar.gz"},
    )
bd.projects.set_current(PROJECT)

if BIOSPHERE not in bd.databases:
    raise RuntimeError(f"Project {PROJECT!r} does not contain {BIOSPHERE!r}.")
if DATABASE in bd.databases:
    raise RuntimeError(f"{DATABASE!r} already exists. Choose a new DATABASE name to import again.")

print(f"Project: {bd.projects.current}")
print(f"Biosphere: {BIOSPHERE} ({len(bd.Database(BIOSPHERE)):,} flows)")

## 3. Import and check links

Use `bw2io.SingleOutputEcospold2Importer` with the five product/UUID-linking strategies listed below. The export already contains the mapped inventory, so ecoinvent-specific cleanup strategies are unnecessary and could replace high uncertainties. The final loop restores the sign flag omitted by the EcoSpold 2 extractor for lognormal exchanges.

In [ ]:
from collections import Counter
from functools import partial
from bw2io.strategies import (
    es2_assign_only_product_with_amount_as_reference_product,
    assign_single_product_as_activity,
    create_composite_code,
    link_biosphere_by_flow_uuid,
    link_internal_technosphere_by_composite_code,
)

importer = bi.SingleOutputEcospold2Importer(
    str(SOURCE),
    DATABASE,
    biosphere_database_name=BIOSPHERE,
    use_mp=False,
    add_product_information=False,
)

# Link the exported UUIDs while preserving the inventory and its uncertainty.
importer.strategies = [
    es2_assign_only_product_with_amount_as_reference_product,
    assign_single_product_as_activity,
    create_composite_code,
    partial(link_biosphere_by_flow_uuid, biosphere=BIOSPHERE),
    link_internal_technosphere_by_composite_code,
]
importer.apply_strategies()
if len(importer.applied_strategies) != len(importer.strategies):
    raise RuntimeError("An EcoSpold 2 linking strategy failed; see the output above.")

# The XML retains signed amounts; stats_arrays also needs this sign flag.
for dataset in importer.data:
    for exchange in dataset["exchanges"]:
        if exchange.get("uncertainty type") == 2:  # Lognormal
            exchange["negative"] = exchange["amount"] < 0

importer.statistics()
unlinked = sum(1 for _ in importer.unlinked)
if unlinked:
    raise RuntimeError(f"{unlinked:,} exchanges remain unlinked; database writing is blocked.")

exchange_counts = Counter(
    exchange["type"]
    for dataset in importer.data
    for exchange in dataset["exchanges"]
)
print(f"Datasets: {len(importer.data):,}")
print(f"Exchanges: {sum(exchange_counts.values()):,}")
print(dict(exchange_counts))

## 4. Write the database

The current export contains **11,947 datasets** and **417,072 exchanges**, with zero unlinked exchanges. This cell creates the inventory database in the selected project.

In [ ]:
if bd.projects.current != PROJECT:
    raise RuntimeError("The current project changed. Rerun the project and import cells.")
if importer.db_name != DATABASE:
    raise RuntimeError("The database name changed. Rerun the import cell before writing.")
if DATABASE in bd.databases:
    raise RuntimeError(f"Refusing to overwrite {DATABASE!r}.")
if any(True for _ in importer.unlinked):
    raise RuntimeError("Resolve unlinked exchanges before writing the database.")

database = importer.write_database()
print(f"Created {database.name!r}: {len(database):,} activities in {bd.projects.current!r}.")

For the export conventions, preserved full names, exclusions and audit fields, see [the export instructions](../docs/bafu-2026-ecospold2-export.md).